# **KCC Preprocessing Notebook**
### **Introduction**
This notebook preprocesses Kisan Call Centre (KCC) Q&A data for the RAG (Retrieval-Augmented Generation) pipeline. The data contains farmer queries and expert responses related to all crops in Uttar Pradesh from 2020-2025.

#### **Preprocessing Steps Covered:**

* Load data
* Filter for agronomic categories
* Handle missing values
* Remove duplicates
* Clean text (remove PII, normalize)
* Add metadata tagging
* Create chunks for RAG
* Save processed data
* Generate metadata schema

**Output:** Cleaned CSV, chunked JSONL (ready for MuRIL embeddings), and metadata schema.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import re
import json
import unicodedata
from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')

## **Step 1: Load Data**
Load the combined KCC dataset from Google Drive. The data contains approximately 3.1 million records for Uttar Pradesh from 2020-2025.

In [58]:
print("\n" + "="*80)
print("STEP 1: LOADING DATA")
print("="*80)

KCC_PATH = "/content/drive/MyDrive/kcc_raw/"
PROCESSED_PATH = "data/processed/kcc/"
FINAL_PATH = "data/final/kcc/"

Path(PROCESSED_PATH).mkdir(parents=True, exist_ok=True)
Path(FINAL_PATH).mkdir(parents=True, exist_ok=True)

data_file = f"{KCC_PATH}/kcc_combined_2020_2025.csv"
print(f"Loading data from: {data_file}")

try:
    df = pd.read_csv(data_file)
    print(f"Loaded {len(df):,} records")
    print(f"Columns: {len(df.columns)}")
except Exception as e:
    print(f"Error loading data: {e}")
    df = pd.read_csv("kcc_combined_2020_2025.csv")
    print(f"Loaded from local: {len(df):,} records")

print("\nFirst 5 rows:")
display(df.head())

print("\nData Info:")
print(df.info())


STEP 1: LOADING DATA
Loading data from: /content/drive/MyDrive/kcc_raw//kcc_combined_2020_2025.csv
Loaded 3,123,029 records
Columns: 15

First 5 rows:


,BlockName,Category,CreatedOn,Crop,DistrictName,KCCCallID,KccAns,QueryText,QueryType,Season,Sector,StateName,day,month,year
0,NAWABGANJ,Others,2020-07-23T15:21:41.12,Others,GONDA,2224700,श्रीमान जी प्रधानमंत्री किसान सम्मान निधि योजन...,Information about application status of PM Kis...,Government Schemes,NaN,AGRICULTURE,UTTAR PRADESH,23.0,7.0,2020.0
1,GHAZIPUR,Vegetables,2020-07-23T15:22:02.727,Cowpea (Vegetable),GHAZIPUR,2225064,--सर आप लोबिया की फसल में Dimethoate 30% EC @...,Give information about plant protection of ...,\tPlant Protection\t,NaN,HORTICULTURE,UTTAR PRADESH,23.0,7.0,2020.0
2,MAHMUDABAD,Cereals,2020-07-25T09:45:10.843,Paddy (Dhan),SITAPUR,2240020,"महोदय, धान में टॉप ड्रेसिंग के समय यूरिया 35 k...",Dhaan ki fasal me top dressing ke samay kya pr...,Nutrient Management,NaN,AGRICULTURE,UTTAR PRADESH,25.0,7.0,2020.0
3,KHUDAGANJ KATRA,Cereals,2020-07-25T09:48:24.863,Paddy (Dhan),SHAHJAHANPUR,2241298,"श्रीमान जी धान की फसल में तना बेधक, पत्ती लपेट...",Information about pest management of leaf fold...,\tPlant Protection\t,NaN,AGRICULTURE,UTTAR PRADESH,25.0,7.0,2020.0
4,BILASPUR,Animal,2020-07-25T09:49:39.7,"Bovine(Cow,Buffalo)",RAMPUR,2240697,श्री मान जी आप डेरी स्कीम की जानकारी के लिए जि...,Dairy scheem ki jankari de…?,Disease Management,NaN,ANIMAL HUSBANDRY,UTTAR PRADESH,25.0,7.0,2020.0



Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3123029 entries, 0 to 3123028
Data columns (total 15 columns):
 #   Column        Dtype  
---  ------        -----  
 0   BlockName     object 
 1   Category      object 
 2   CreatedOn     object 
 3   Crop          object 
 4   DistrictName  object 
 5   KCCCallID     int64  
 6   KccAns        object 
 7   QueryText     object 
 8   QueryType     object 
 9   Season        float64
 10  Sector        object 
 11  StateName     object 
 12  day           float64
 13  month         float64
 14  year          float64
dtypes: float64(4), int64(1), object(10)
memory usage: 357.4+ MB
None


## **Step 2: Data Overview & Quality Check**
Examine the dataset structure, data types, and verify the filters (UP state and years 2020-2025).

In [59]:
print("\n" + "="*80)
print("STEP 2: DATA OVERVIEW & QUALITY CHECK")
print("="*80)

print("\n2.1 Columns in dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n2.2 Data types:")
print(df.dtypes)

print("\n2.3 Basic statistics for numeric columns:")
print(df.describe())

print("\n2.4 Verifying filters...")
if 'StateName' in df.columns:
    states = df['StateName'].unique()
    print(f"  States in data: {states}")
    if 'UTTAR PRADESH' in states:
        print("  Data is filtered for Uttar Pradesh")
    else:
        print("  Data may not be filtered for UP")

if 'year' in df.columns:
    years = sorted(df['year'].unique())
    print(f"  Years in data: {years}")
    if min(years) >= 2020:
        print("  Data is filtered for years >= 2020")

if 'Crop' in df.columns:
    print("\n2.5 Crop distribution (top 10):")
    print(df['Crop'].value_counts().head(10))


STEP 2: DATA OVERVIEW & QUALITY CHECK

2.1 Columns in dataset:
   1. BlockName
   2. Category
   3. CreatedOn
   4. Crop
   5. DistrictName
   6. KCCCallID
   7. KccAns
   8. QueryText
   9. QueryType
  10. Season
  11. Sector
  12. StateName
  13. day
  14. month
  15. year

2.2 Data types:
BlockName        object
Category         object
CreatedOn        object
Crop             object
DistrictName     object
KCCCallID         int64
KccAns           object
QueryText        object
QueryType        object
Season          float64
Sector           object
StateName        object
day             float64
month           float64
year            float64
dtype: object

2.3 Basic statistics for numeric columns:
          KCCCallID  Season           day         month          year
count  3.123029e+06     0.0  3.123027e+06  3.123027e+06  3.123027e+06
mean   3.364788e+06     NaN  1.563869e+01  6.093401e+00  2.022317e+03
std    2.101280e+06     NaN  8.854212e+00  3.599894e+00  1.605579e+00
min    2.

## **Step 3: Filter for Agronomic Categories**
Keep only agronomic queries and exclude financial, subsidy, and market-related entries as per Milestone 1 risk mitigation.

In [71]:
print("\n" + "="*80)
print("STEP 3: FILTERING FOR AGRONOMIC CATEGORIES")
print("="*80)

initial_count = len(df)

agri_categories = [
    'Cereals', 'Pulses', 'Oilseeds', 'Vegetables', 'Fruits',
    'Plant Protection', 'Nutrient Management', 'Fertilizer Management',
    'Irrigation Management', 'Weed Management', 'Pest Management',
    'Disease Management', 'Soil Management', 'Crop Management',
    'Variety Selection', 'Seed Management', 'Harvesting', 'Sowing',
    'Crop Protection', 'Fertilizer Use', 'Water Management'
]

exclude_categories = [
    'Government Schemes', 'Market Information', 'Crop Insurance',
    'Credit', 'Subsidy', 'Finance', 'Banking', 'Insurance',
    'Weather', 'Climate', 'Rainfall'
]

if 'Category' in df.columns:
    agri_mask = df['Category'].str.contains('|'.join(agri_categories), case=False, na=False)
    df_filtered = df[agri_mask]

    exclude_mask = df_filtered['Category'].str.contains('|'.join(exclude_categories), case=False, na=False)
    df_filtered = df_filtered[~exclude_mask]

    print(f"Records after agronomic filter: {len(df_filtered):,} ({len(df_filtered)/initial_count*100:.2f}%)")
    print("\nRemaining categories:")
    print(df_filtered['Category'].value_counts().head(10))
else:
    print("'Category' column not found. Skipping filter.")
    df_filtered = df.copy()


STEP 3: FILTERING FOR AGRONOMIC CATEGORIES
Records after agronomic filter: 1,701,442 (54.48%)

Remaining categories:
Category
Cereals                      1000881
Vegetables                    336545
Oilseeds                      153852
Pulses                        109358
Fruits                        100707
Fruiting Vegetables Crops         44
Tropical Fruits                   23
Sub - Tropical Fruits             19
Root Vegetables Crops              8
Temperate Fruits                   4
Name: count, dtype: int64


##**Step 4: Handle Missing Values**
This step addresses missing data in the dataset. For critical fields (QueryText, KccAns), we drop records as these cannot be inferred. For Crop, we fill missing values with 'Unknown' since all crops are retained and crop filtering is not required. For District, we fill missing values with 'Unknown'. Other less critical fields (Season, QueryType, BlockName) are filled with default values.

In [72]:
print("\n" + "="*80)
print("STEP 4: CHECKING & HANDLING MISSING VALUES")
print("="*80)

print("\nMissing values count per column:")
missing_counts = df_filtered.isnull().sum()
missing_pct = (missing_counts / len(df_filtered)) * 100

missing_df = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing_Count': missing_counts.values,
    'Missing_%': missing_pct.values
}).sort_values('Missing_%', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0].to_string(index=False))

print("\n" + "="*60)
print("Summary:")
print("="*60)
print(f"Total records: {len(df_filtered):,}")
print(f"Columns with missing values: {(missing_counts > 0).sum()}")
print(f"Total missing cells: {missing_counts.sum():,}")

print("\n" + "="*60)
print("Dropping Unnecessary Columns...")
print("="*60)

columns_to_drop = [
    'CreatedOn',      # Redundant (year/month/day exist)
    'Sector',         # Only Agriculture
    'StateName',      # All records are Uttar Pradesh
    'KCCCallID',      # Not needed for RAG
    'day',            # Not needed
    'BlockName'       # We have district
]


# Drop columns that exist
columns_to_drop = [col for col in columns_to_drop if col in df_filtered.columns]
df_filtered = df_filtered.drop(columns=columns_to_drop)

print(f"Dropped columns: {columns_to_drop}")
print(f"Remaining columns: {df_filtered.columns.tolist()}")

print("\n" + "="*60)
print("Handling Missing Values...")
print("="*60)


print("\n1. Critical Fields (QueryText & KccAns):")
print("-" * 40)

if 'QueryText' in df_filtered.columns:
    before = len(df_filtered)
    df_filtered = df_filtered[df_filtered['QueryText'].notna()]
    removed = before - len(df_filtered)
    print(f"  Removed {removed:,} records with missing QueryText")

if 'KccAns' in df_filtered.columns:
    before = len(df_filtered)
    df_filtered = df_filtered[df_filtered['KccAns'].notna()]
    removed = before - len(df_filtered)
    print(f"  Removed {removed:,} records with missing KccAns")

print(f"  Remaining records: {len(df_filtered):,}")

print("\n2. Crop Column (fill with 'Unknown'):")
print("-" * 40)

if 'Crop' in df_filtered.columns:
    missing_crop = df_filtered['Crop'].isna().sum()
    if missing_crop > 0:
        df_filtered['Crop'] = df_filtered['Crop'].fillna('Unknown')
        print(f"  Filled {missing_crop:,} missing Crop with 'Unknown'")
    else:
        print("  No missing Crop values found")

print("\n3. District Column (fill with 'Unknown'):")
print("-" * 40)

if 'DistrictName' in df_filtered.columns:
    missing_district = df_filtered['DistrictName'].isna().sum()
    if missing_district > 0:
        df_filtered['DistrictName'] = df_filtered['DistrictName'].fillna('Unknown')
        print(f"  Filled {missing_district:,} missing District with 'Unknown'")
    else:
        print("  No missing District values found")


print("\n4. Season Column (infer from month):")
print("-" * 40)

if 'Season' in df_filtered.columns:
    if 'month' in df_filtered.columns:
        def infer_season(month):
            if pd.isna(month):
                return 'Unknown'
            if month in [6, 7, 8, 9, 10]:
                return 'Kharif'
            elif month in [11, 12, 1, 2, 3]:
                return 'Rabi'
            else:
                return 'Zaid'

        before = df_filtered['Season'].isna().sum()
        df_filtered['Season'] = df_filtered.apply(
            lambda row: infer_season(row['month']) if pd.isna(row['Season']) else row['Season'],
            axis=1
        )
        after = df_filtered['Season'].isna().sum()
        print(f"  Inferred Season from month: {before - after:,} records filled")
        print(f"  Remaining missing: {after:,} (will fill with 'Unknown')")
        df_filtered['Season'] = df_filtered['Season'].fillna('Unknown')
    else:
        before = df_filtered['Season'].isna().sum()
        df_filtered['Season'] = df_filtered['Season'].fillna('Unknown')
        print(f"  Filled {before:,} missing Season with 'Unknown'")


print("\5. QueryType Column (fill with 'Other'):")
print("-" * 40)

if 'QueryType' in df_filtered.columns:
    missing_qtype = df_filtered['QueryType'].isna().sum()
    if missing_qtype > 0:
        df_filtered['QueryType'] = df_filtered['QueryType'].fillna('Other')
        print(f"  Filled {missing_qtype:,} missing QueryType with 'Other'")
    else:
        print("  No missing QueryType values found")

if 'month' in df_filtered.columns:
    before = len(df_filtered)
    df_filtered = df_filtered[df_filtered['month'].notna()]
    removed = before - len(df_filtered)
    print(f"  Removed {removed:,} records with missing month")

if 'year' in df_filtered.columns:
    before = len(df_filtered)
    df_filtered = df_filtered[df_filtered['year'].notna()]
    removed = before - len(df_filtered)
    print(f"  Removed {removed:,} records with missing year")

print("\n" + "="*60)
print("Verification - No Critical Missing Values Should Remain")
print("="*60)

critical_fields = ['QueryText', 'KccAns', 'Crop', 'DistrictName']
for field in critical_fields:
    if field in df_filtered.columns:
        missing = df_filtered[field].isnull().sum()
        status = "OK" if missing == 0 else f"WARNING: {missing} missing"
        print(f"  {field}: {status}")

print("\n" + "="*60)
print("Final Summary")
print("="*60)
print(f"Records after handling missing values: {len(df_filtered):,}")
print(f"Columns remaining: {df_filtered.columns.tolist()}")


STEP 4: CHECKING & HANDLING MISSING VALUES

Missing values count per column:
      Column  Missing_Count  Missing_%
      Season        1701442 100.000000
      Sector           1916   0.112610
        Crop           1914   0.112493
   QueryType            800   0.047019
   BlockName            335   0.019689
      KccAns            111   0.006524
   QueryText              7   0.000411
   StateName              2   0.000118
         day              2   0.000118
       month              2   0.000118
        year              2   0.000118
   CreatedOn              1   0.000059
DistrictName              1   0.000059

Summary:
Total records: 1,701,442
Columns with missing values: 13
Total missing cells: 1,706,535

Dropping Unnecessary Columns...
Dropped columns: ['CreatedOn', 'Sector', 'StateName', 'KCCCallID', 'day', 'BlockName']
Remaining columns: ['Category', 'Crop', 'DistrictName', 'KccAns', 'QueryText', 'QueryType', 'Season', 'month', 'year']

Handling Missing Values...

1. Critica

In [74]:
print("\n" + "="*60)
print("Verification - No Missing Values Should Remain")
print("="*60)

remaining_missing = df_filtered.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) == 0:
    print("✅ All missing values handled successfully!")
    print(f"   Final records: {len(df_filtered):,}")
else:
    print("⚠️ Still have missing values:")
    print(remaining_missing)


Verification - No Missing Values Should Remain
✅ All missing values handled successfully!
   Final records: 1,701,322


## **Step 5: Remove Duplicates**
Remove duplicate Q&A pairs for the same crop. Identical queries with different answers or contexts are preserved for RAG diversity.

In [75]:
print("\n" + "="*80)
print("STEP 5: DUPLICATE REMOVAL")
print("="*80)

print(f"Records before deduplication: {len(df_filtered):,}")

print("\n" + "-"*60)
print("Removing exact duplicates (all columns)...")
print("-"*60)

before = len(df_filtered)
df_filtered = df_filtered.drop_duplicates(keep='first')
removed = before - len(df_filtered)
print(f"  Removed: {removed:,} exact duplicates")
print(f"  Remaining: {len(df_filtered):,}")

print("\n" + "-"*60)
print("Removing duplicate Q&A pairs for same crop...")
print("-"*60)

qa_cols = ['QueryText', 'KccAns', 'Crop']
qa_cols = [col for col in qa_cols if col in df_filtered.columns]

before = len(df_filtered)
df_filtered = df_filtered.drop_duplicates(subset=qa_cols, keep='first')
removed = before - len(df_filtered)
print(f"  Removed: {removed:,} duplicate Q&A pairs for same crop")
print(f"  Remaining: {len(df_filtered):,}")

print("\n" + "="*60)
print("Summary")
print("="*60)
print(f"  Records before dedup: {before:,}")
print(f"  Records after dedup:  {len(df_filtered):,}")
print(f"  Removed:              {before - len(df_filtered):,} duplicates")
print(f"  Reduction:            {(1 - len(df_filtered)/before)*100:.1f}%")


STEP 5: DUPLICATE REMOVAL
Records before deduplication: 1,701,322

------------------------------------------------------------
Removing exact duplicates (all columns)...
------------------------------------------------------------
  Removed: 39,111 exact duplicates
  Remaining: 1,662,211

------------------------------------------------------------
Removing duplicate Q&A pairs for same crop...
------------------------------------------------------------
  Removed: 202,509 duplicate Q&A pairs for same crop
  Remaining: 1,459,702

Summary
  Records before dedup: 1,662,211
  Records after dedup:  1,459,702
  Removed:              202,509 duplicates
  Reduction:            12.2%


## **Step 6: Clean Text**
Clean query and answer texts by removing PII, fixing encoding issues, normalizing Indian scripts, and detecting languages.

In [76]:
print("\n" + "="*80)
print("STEP 6: TEXT CLEANING")
print("="*80)

def clean_text(text):
    if pd.isna(text) or text == '':
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('Ã', '').replace('Â', '')
    text = text.replace('â', "'").replace('â', '"').replace('â', '"')
    text = re.sub(r'[^\w\s\u0900-\u097F\u0B80-\u0BFF\u0C00-\u0C7F\u0D00-\u0D7F.,!?\'"()-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_pii(text):
    if pd.isna(text) or text == '':
        return text
    text = re.sub(r'\b\d{10}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL]', text)
    text = re.sub(r'\b[A-Z0-9]{8,}\b', '[ID]', text)
    return text

def normalize_indic_text(text):
    if pd.isna(text) or text == '':
        return text
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[\u200B-\u200D\uFEFF]', '', text)
    return text

def detect_language(text):
    if pd.isna(text) or text == '' or len(str(text)) < 3:
        return 'unknown'
    text = str(text)
    if re.search(r'[\u0900-\u097F]', text):
        devanagari_chars = len(re.findall(r'[\u0900-\u097F]', text))
        total_chars = len(re.sub(r'\s', '', text))
        if total_chars > 0 and devanagari_chars / total_chars > 0.3:
            return 'hindi'
    english_chars = len(re.findall(r'[a-zA-Z]', text))
    total_chars = len(re.sub(r'\s', '', text))
    if total_chars > 0 and english_chars / total_chars > 0.5:
        return 'english'
    return 'mixed'

print("Cleaning QueryText and KccAns...")
df_filtered['cleaned_query'] = df_filtered['QueryText'].apply(clean_text)
df_filtered['cleaned_answer'] = df_filtered['KccAns'].apply(clean_text)

print("Removing PII...")
df_filtered['cleaned_query'] = df_filtered['cleaned_query'].apply(remove_pii)
df_filtered['cleaned_answer'] = df_filtered['cleaned_answer'].apply(remove_pii)

print("Normalizing Indian text...")
df_filtered['cleaned_query'] = df_filtered['cleaned_query'].apply(normalize_indic_text)
df_filtered['cleaned_answer'] = df_filtered['cleaned_answer'].apply(normalize_indic_text)

print("Detecting languages...")
df_filtered['query_lang'] = df_filtered['cleaned_query'].apply(detect_language)
df_filtered['answer_lang'] = df_filtered['cleaned_answer'].apply(detect_language)

print("\nLanguage distribution:")
print("  Query languages:")
for lang, count in df_filtered['query_lang'].value_counts().items():
    pct = count / len(df_filtered) * 100
    print(f"    {lang}: {count:,} ({pct:.1f}%)")

print("\nRemoving records with very short text...")
before = len(df_filtered)
df_filtered = df_filtered[
    (df_filtered['cleaned_query'].str.len() > 5) |
    (df_filtered['cleaned_answer'].str.len() > 5)
]
print(f"  Removed: {before - len(df_filtered):,} records")
print(f"  Remaining: {len(df_filtered):,}")


STEP 6: TEXT CLEANING
Cleaning QueryText and KccAns...
Removing PII...
Normalizing Indian text...
Detecting languages...

Language distribution:
  Query languages:
    english: 1,459,452 (100.0%)
    mixed: 243 (0.0%)
    unknown: 7 (0.0%)

Removing records with very short text...
  Removed: 10 records
  Remaining: 1,459,692


## **Step 8: Add Metadata**
Add structured metadata to each record for filtered retrieval in the RAG pipeline. Metadata includes crop, district, season, query type, and language information.

In [77]:
print("\n" + "="*80)
print("STEP 7: METADATA TAGGING")
print("="*80)

print("Creating metadata columns...")
df_filtered['metadata'] = df_filtered.apply(lambda row: {
    'crop': row.get('Crop', 'unknown'),
    'district': row.get('DistrictName', 'unknown'),
    'block': row.get('BlockName', 'unknown'),
    'season': row.get('Season', 'unknown'),
    'query_type': row.get('QueryType', 'other'),
    'category': row.get('Category', 'other'),
    'year': int(row.get('year', 0)) if pd.notna(row.get('year')) else 0,
    'month': int(row.get('month', 0)) if pd.notna(row.get('month')) else 0,
    'language': row.get('query_lang', 'unknown')
}, axis=1)

print("Metadata sample:")
print(json.dumps(df_filtered['metadata'].iloc[0], indent=2, ensure_ascii=False))


STEP 7: METADATA TAGGING
Creating metadata columns...
Metadata sample:
{
  "crop": "Cowpea (Vegetable)",
  "district": "GHAZIPUR",
  "block": "unknown",
  "season": "Kharif",
  "query_type": "\tPlant Protection\t",
  "category": "Vegetables",
  "year": 2020,
  "month": 7,
  "language": "english"
}


##**Step 9: Create Chunks**
Split long Q&A pairs into chunks suitable for embedding (target: 256-512 tokens). This ensures compatibility with MuRIL's 512 token limit while preserving context through overlap.

###**Chunking Strategy:**
**Chunk Size:** 512 characters (MuRIL's token limit is 512)

**Overlap:** 50 characters (preserves context across chunks)

**Format:** "Question: {query}\nAnswer: {answer}"

**Preserve Sentences:** Split at sentence boundaries (.!?) to maintain meaning

**Single Chunk:** If text fits in 512 chars, keep as one chunk

**Multiple Chunks:** Split long texts at sentence boundaries with overlap

In [81]:
print("\n" + "="*80)
print("STEP 9: CHUNK PREPARATION")
print("="*80)

CHUNK_SIZE = 512
OVERLAP = 50

print(f"Chunk Configuration:")
print(f"  Chunk size: {CHUNK_SIZE} characters")
print(f"  Overlap: {OVERLAP} characters")
print(f"  Format: Question: {{query}}\nAnswer: {{answer}}")
print(f"  Split Strategy: Sentence boundaries (.!?)")
print(f"  Target: {CHUNK_SIZE - 100}-{CHUNK_SIZE} chars (MuRIL limit)")

def create_chunks(row):
    """
    Create RAG chunks from Q&A pairs.

    Strategy:
    1. Combine Query and Answer with labels
    2. If text fits in CHUNK_SIZE, return single chunk
    3. If longer, split at sentence boundaries (.!?)
    4. Apply overlap between chunks to preserve context
    5. Each chunk carries full metadata for filtered retrieval
    """
    query = row.get('cleaned_query', '')
    answer = row.get('cleaned_answer', '')

    if not query and not answer:
        return []

    qa_text = f"Question: {query}\nAnswer: {answer}"

    # Single chunk - fits within limit
    if len(qa_text) <= CHUNK_SIZE:
        return [{
            'text': qa_text,
            'metadata': row.get('metadata', {}),
            'chunk_number': 1,
            'total_chunks': 1
        }]

    # Multiple chunks - split at sentence boundaries
    chunks = []
    sentences = re.split(r'(?<=[.!?])\s+', qa_text)
    current_chunk = ""
    chunk_num = 1

    for sentence in sentences:
        # If adding sentence exceeds CHUNK_SIZE, save current chunk
        if len(current_chunk) + len(sentence) + 1 > CHUNK_SIZE:
            if current_chunk:
                chunks.append({
                    'text': current_chunk.strip(),
                    'metadata': row.get('metadata', {}),
                    'chunk_number': chunk_num,
                    'total_chunks': 0
                })
                chunk_num += 1

                # Start new chunk with overlap (last 3-5 words)
                if OVERLAP > 0:
                    words = current_chunk.split()
                    if len(words) > 5:
                        overlap_text = ' '.join(words[-5:])
                        current_chunk = overlap_text + ' ' + sentence
                    elif len(words) > 3:
                        overlap_text = ' '.join(words[-3:])
                        current_chunk = overlap_text + ' ' + sentence
                    else:
                        current_chunk = sentence
                else:
                    current_chunk = sentence
            else:
                current_chunk = sentence
        else:
            current_chunk += ' ' + sentence if current_chunk else sentence

    # Add the last chunk
    if current_chunk:
        chunks.append({
            'text': current_chunk.strip(),
            'metadata': row.get('metadata', {}),
            'chunk_number': chunk_num,
            'total_chunks': 0
        })

    # Update total_chunks for all chunks
    total = len(chunks)
    for chunk in chunks:
        chunk['total_chunks'] = total

    return chunks

print("\nCreating chunks...")
all_chunks = []
processed = 0
failed = 0

batch_size = 10000
num_batches = (len(df_filtered) + batch_size - 1) // batch_size
print(f"Processing {len(df_filtered):,} records in {num_batches} batches...")

for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(df_filtered))

    for idx in range(start_idx, end_idx):
        try:
            row = df_filtered.iloc[idx]
            chunks = create_chunks(row)
            if chunks:
                all_chunks.extend(chunks)
            processed += 1
        except Exception as e:
            failed += 1
            if failed <= 5:
                print(f"  Error at row {idx}: {e}")

    if (batch_idx + 1) % 5 == 0:
        print(f"  Batch {batch_idx + 1}/{num_batches} ({end_idx:,} records) - Chunks: {len(all_chunks):,}")

print(f"\nChunking complete!")
print(f"  Records processed: {processed:,}")
print(f"  Chunks created: {len(all_chunks):,}")
print(f"  Failed: {failed}")

print("\nChunk statistics:")
if all_chunks:
    chunk_lengths = [len(c['text']) for c in all_chunks]
    print(f"  Total chunks: {len(all_chunks):,}")
    print(f"  Average length: {sum(chunk_lengths)/len(chunk_lengths):.0f} characters")
    print(f"  Min length: {min(chunk_lengths)}")
    print(f"  Max length: {max(chunk_lengths)}")

    single_chunk = sum(1 for c in all_chunks if c['total_chunks'] == 1)
    multi_chunk = len(all_chunks) - single_chunk
    print(f"  Single chunk records: {single_chunk:,} ({single_chunk/len(all_chunks)*100:.1f}%)")
    print(f"  Multi-chunk records: {multi_chunk:,} ({multi_chunk/len(all_chunks)*100:.1f}%)")


STEP 9: CHUNK PREPARATION
Chunk Configuration:
  Chunk size: 512 characters
  Overlap: 50 characters
  Format: Question: {query}
Answer: {answer}
  Split Strategy: Sentence boundaries (.!?)
  Target: 412-512 chars (MuRIL limit)

Creating chunks...
Processing 1,459,692 records in 146 batches...
  Batch 5/146 (50,000 records) - Chunks: 50,145
  Batch 10/146 (100,000 records) - Chunks: 100,290
  Batch 15/146 (150,000 records) - Chunks: 150,443
  Batch 20/146 (200,000 records) - Chunks: 200,553
  Batch 25/146 (250,000 records) - Chunks: 250,673
  Batch 30/146 (300,000 records) - Chunks: 300,814
  Batch 35/146 (350,000 records) - Chunks: 351,158
  Batch 40/146 (400,000 records) - Chunks: 401,554
  Batch 45/146 (450,000 records) - Chunks: 451,939
  Batch 50/146 (500,000 records) - Chunks: 502,384
  Batch 55/146 (550,000 records) - Chunks: 552,790
  Batch 60/146 (600,000 records) - Chunks: 603,215
  Batch 65/146 (650,000 records) - Chunks: 653,646
  Batch 70/146 (700,000 records) - Chunks: 7

##**Chunk Preview**

Preview the created chunks to verify quality, distribution, and structure before saving.

In [83]:
print("\n" + "="*80)
print("CHUNK PREVIEW")
print("="*80)

if len(all_chunks) > 0:

    print("\n1. SAMPLE CHUNKS:")
    print("-" * 60)

    for i, chunk in enumerate(all_chunks[:3], 1):
        print(f"\nChunk {i}:")
        print(f"  Crop: {chunk['metadata'].get('crop', 'N/A')}")
        print(f"  District: {chunk['metadata'].get('district', 'N/A')}")
        print(f"  Season: {chunk['metadata'].get('season', 'N/A')}")
        print(f"  Year: {chunk['metadata'].get('year', 'N/A')}")
        print(f"  Query Type: {chunk['metadata'].get('query_type', 'N/A')}")
        print(f"  Chunk: {chunk.get('chunk_number', 0)}/{chunk.get('total_chunks', 0)}")
        print(f"  Text: {chunk['text'][:200]}...")
        print("-" * 60)

    print("\n2. COMPLETE CHUNK (JSON):")
    print("-" * 60)
    print(json.dumps(all_chunks[0], indent=2, ensure_ascii=False))

    print("\n3. CHUNK DISTRIBUTION BY CROP:")
    print("-" * 60)
    crop_chunks = {}
    for chunk in all_chunks:
        crop = chunk['metadata'].get('crop', 'unknown')
        crop_chunks[crop] = crop_chunks.get(crop, 0) + 1

    for crop, count in sorted(crop_chunks.items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(all_chunks)) * 100
        print(f"  {crop}: {count:,} chunks ({pct:.1f}%)")

    print("\n4. CHUNK DISTRIBUTION BY LANGUAGE:")
    print("-" * 60)
    lang_chunks = {}
    for chunk in all_chunks:
        lang = chunk['metadata'].get('language', 'unknown')
        lang_chunks[lang] = lang_chunks.get(lang, 0) + 1

    for lang, count in sorted(lang_chunks.items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(all_chunks)) * 100
        print(f"  {lang}: {count:,} chunks ({pct:.1f}%)")

    print("\n5. CHUNK SIZE DISTRIBUTION:")
    print("-" * 60)
    chunk_lengths = [len(c['text']) for c in all_chunks]

    size_ranges = {
        '0-100': 0,
        '101-200': 0,
        '201-300': 0,
        '301-400': 0,
        '401-500': 0,
        '501-512': 0
    }

    for length in chunk_lengths:
        if length <= 100:
            size_ranges['0-100'] += 1
        elif length <= 200:
            size_ranges['101-200'] += 1
        elif length <= 300:
            size_ranges['201-300'] += 1
        elif length <= 400:
            size_ranges['301-400'] += 1
        elif length <= 500:
            size_ranges['401-500'] += 1
        else:
            size_ranges['501-512'] += 1

    for range_name, count in size_ranges.items():
        if count > 0:
            pct = (count / len(all_chunks)) * 100
            print(f"  {range_name} chars: {count:,} chunks ({pct:.1f}%)")

    print("\n6. CHUNK STATISTICS:")
    print("-" * 60)
    print(f"  Total chunks: {len(all_chunks):,}")
    print(f"  Average length: {sum(chunk_lengths)/len(chunk_lengths):.0f} characters")
    print(f"  Min length: {min(chunk_lengths)}")
    print(f"  Max length: {max(chunk_lengths)}")

    single_chunk = sum(1 for c in all_chunks if c['total_chunks'] == 1)
    multi_chunk = len(all_chunks) - single_chunk
    print(f"  Single chunk records: {single_chunk:,} ({single_chunk/len(all_chunks)*100:.1f}%)")
    print(f"  Multi-chunk records: {multi_chunk:,} ({multi_chunk/len(all_chunks)*100:.1f}%)")

    print("\n7. MULTI-CHUNK EXAMPLE:")
    print("-" * 60)
    multi_chunk_examples = [c for c in all_chunks if c['total_chunks'] > 1]
    if multi_chunk_examples:
        example = multi_chunk_examples[0]
        print(f"  Crop: {example['metadata'].get('crop', 'N/A')}")
        print(f"  Total chunks: {example['total_chunks']}")
        print(f"  Chunk {example['chunk_number']} of {example['total_chunks']}:")
        print(f"  Text: {example['text'][:150]}...")
    else:
        print("  No multi-chunk records found (all records fit in one chunk)")

else:
    print("No chunks created!")


CHUNK PREVIEW

1. SAMPLE CHUNKS:
------------------------------------------------------------

Chunk 1:
  Crop: Cowpea (Vegetable)
  District: GHAZIPUR
  Season: Kharif
  Year: 2020
  Query Type: 	Plant Protection	
  Chunk: 1/1
  Text: Question: Give information about plant protection of Cowpea ?
Answer: --सर आप लोबिया की फसल में Dimethoate 30 EC 400 ml एकर 200 लीटर पानी में घोल बनाकर स्प्रे करे...
------------------------------------------------------------

Chunk 2:
  Crop: Paddy (Dhan)
  District: SITAPUR
  Season: Kharif
  Year: 2020
  Query Type: Nutrient Management
  Chunk: 1/1
  Text: Question: Dhaan ki fasal me top dressing ke samay kya prayog kare ...?
Answer: महोदय, धान में टॉप ड्रेसिंग के समय यूरिया 35 kg और जिंक सल्फेट 10 kg प्रति एकर की दर से नमी की अवस्था में प्रयोग करे...
------------------------------------------------------------

Chunk 3:
  Crop: Paddy (Dhan)
  District: SHAHJAHANPUR
  Season: Kharif
  Year: 2020
  Query Type: 	Plant Protection	
  Chunk: 1/1
  Text: 

##**Step 10: Save Processed Data**
Save the processed data to Google Drive with organized folder structure for processed and final datasets.

**Processed Folder (kcc_raw/processed/)** - Contains intermediate data files:

**kcc_cleaned_all_crops.csv:** Final cleaned dataset after all preprocessing steps (filtering, missing value handling, deduplication, text cleaning). Contains all records with cleaned text and metadata. Size: 1.99 GB

**kcc_chunks_sample_1000.jsonl:** Sample of 1000 chunks for quick testing and validation.Size:762 KB

**kcc_qa_pairs.csv:** Query-Answer pairs in CSV format for easy viewing and manual review. Contains cleaned queries, answers, and key metadata. Size: 893.7 MB

**Final Folder (kcc_processed/final/)** - Contains RAG-ready data:

**kcc_chunks_rag.jsonl:** Main chunked dataset ready for MuRIL embedding. Each record contains Q&A text split into 512-character chunks with complete metadata for filtered retrieval. Size: 1.18 GB

**metadata_schema.json:** Complete documentation of dataset structure, chunk configuration, metadata schema, embedding model specifications, and RAG tiered relevance thresholds. Size:7 KB

In [84]:
print("\n" + "="*80)
print("STEP 9: SAVING PROCESSED DATA TO GOOGLE DRIVE")
print("="*80)

# Define Google Drive paths

DRIVE_PROCESSED = f"{KCC_PATH}processed/"
DRIVE_FINAL = f"{KCC_PATH}final/"

# Create directories
!mkdir -p "{DRIVE_PROCESSED}"
!mkdir -p "{DRIVE_FINAL}"

print(f"Google Drive folders created:")
print(f"  Processed: {DRIVE_PROCESSED}")
print(f"  Final: {DRIVE_FINAL}")

# Helper function to convert numpy types to Python native types
def convert_to_native(obj):
    if isinstance(obj, dict):
        return {k: convert_to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_native(v) for v in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    else:
        return obj

print("\n" + "-"*60)
print("1. Saving Cleaned CSV (Processed Data)")
print("-"*60)

# Save to Google Drive - Processed folder
cleaned_path_drive = f"{DRIVE_PROCESSED}/kcc_cleaned_all_crops.csv"
df_filtered.to_csv(cleaned_path_drive, index=False)
print(f"  Saved to Drive: {cleaned_path_drive}")
print(f"  Records: {len(df_filtered):,}")

# Also save locally
cleaned_path_local = f"{PROCESSED_PATH}/kcc_cleaned_all_crops.csv"
df_filtered.to_csv(cleaned_path_local, index=False)
print(f"  Saved locally: {cleaned_path_local}")

print("\n" + "-"*60)
print("2. Saving Chunks JSONL (Final Data - Ready for MuRIL)")
print("-"*60)

# Save to Google Drive - Final folder
chunks_path_drive = f"{DRIVE_FINAL}/kcc_chunks_rag.jsonl"
with open(chunks_path_drive, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
print(f"  Saved to Drive: {chunks_path_drive}")
print(f"  Chunks: {len(all_chunks):,}")

# Also save locally
chunks_path_local = f"{FINAL_PATH}/kcc_chunks_rag.jsonl"
with open(chunks_path_local, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
print(f"  Saved locally: {chunks_path_local}")

print("\n" + "-"*60)
print("3. Saving Metadata Schema")
print("-"*60)

# Use existing metadata_schema from previous cell
# Update with final counts
metadata_schema['total_records'] = int(len(df_filtered))
metadata_schema['total_chunks'] = int(len(all_chunks))
metadata_schema['language_distribution'] = convert_to_native(dict(df_filtered['query_lang'].value_counts()))
metadata_schema['crop_distribution'] = convert_to_native(dict(df_filtered['Crop'].value_counts()))
metadata_schema['query_type_distribution'] = convert_to_native(dict(df_filtered['QueryType'].value_counts().head(10)))

# Save to Google Drive - Final folder
metadata_path_drive = f"{DRIVE_FINAL}/metadata_schema.json"
with open(metadata_path_drive, 'w', encoding='utf-8') as f:
    json.dump(metadata_schema, f, indent=2, ensure_ascii=False)
print(f"  Saved to Drive: {metadata_path_drive}")

# Also save locally
metadata_path_local = f"{FINAL_PATH}/metadata_schema.json"
with open(metadata_path_local, 'w', encoding='utf-8') as f:
    json.dump(metadata_schema, f, indent=2, ensure_ascii=False)
print(f"  Saved locally: {metadata_path_local}")

print("\n" + "-"*60)
print("4. Saving Sample Chunks (For Quick Testing)")
print("-"*60)

sample_chunks = all_chunks[:1000]

# Save to Google Drive - Processed folder
sample_chunks_path_drive = f"{DRIVE_PROCESSED}/kcc_chunks_sample_1000.jsonl"
with open(sample_chunks_path_drive, 'w', encoding='utf-8') as f:
    for chunk in sample_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
print(f"  Saved to Drive: {sample_chunks_path_drive}")

# Also save locally
sample_chunks_path_local = f"{PROCESSED_PATH}/kcc_chunks_sample_1000.jsonl"
with open(sample_chunks_path_local, 'w', encoding='utf-8') as f:
    for chunk in sample_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
print(f"  Saved locally: {sample_chunks_path_local}")

print("\n" + "-"*60)
print("5. Saving Q&A Pairs (For Easy Viewing)")
print("-"*60)

# Only use columns that exist in the dataframe
qa_cols = ['cleaned_query', 'cleaned_answer', 'Crop', 'DistrictName', 'QueryType', 'year', 'Season']
qa_cols = [col for col in qa_cols if col in df_filtered.columns]

if len(qa_cols) > 0:
    qa_df = df_filtered[qa_cols]

    # Save to Google Drive - Processed folder
    qa_path_drive = f"{DRIVE_PROCESSED}/kcc_qa_pairs.csv"
    qa_df.to_csv(qa_path_drive, index=False)
    print(f"  Saved to Drive: {qa_path_drive}")
    print(f"  Records: {len(qa_df):,}")
    print(f"  Columns: {qa_df.columns.tolist()}")

    # Also save locally
    qa_path_local = f"{PROCESSED_PATH}/kcc_qa_pairs.csv"
    qa_df.to_csv(qa_path_local, index=False)
    print(f"  Saved locally: {qa_path_local}")
else:
    print("  No Q&A columns available to save")

print("\n" + "="*60)
print("Files Saved Successfully!")
print("="*60)




STEP 9: SAVING PROCESSED DATA TO GOOGLE DRIVE
Google Drive folders created:
  Processed: /content/drive/MyDrive/kcc_raw/processed/
  Final: /content/drive/MyDrive/kcc_raw/final/

------------------------------------------------------------
1. Saving Cleaned CSV (Processed Data)
------------------------------------------------------------
  Saved to Drive: /content/drive/MyDrive/kcc_raw/processed//kcc_cleaned_all_crops.csv
  Records: 1,459,692
  Saved locally: data/processed/kcc//kcc_cleaned_all_crops.csv

------------------------------------------------------------
2. Saving Chunks JSONL (Final Data - Ready for MuRIL)
------------------------------------------------------------
  Saved to Drive: /content/drive/MyDrive/kcc_raw/final//kcc_chunks_rag.jsonl
  Chunks: 1,468,625
  Saved locally: data/final/kcc//kcc_chunks_rag.jsonl

------------------------------------------------------------
3. Saving Metadata Schema
------------------------------------------------------------
  Saved to D